In [1]:
!pip install requests beautifulsoup4 lxml cloudscraper google-generativeai -q
!pip install firecrawl-py pandas pydantic google-generativeai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.0/233.0 kB 4.8 MB/s eta 0:00:00


In [2]:
import numpy as np 
import pandas as pd 
import os
import time
from firecrawl import FirecrawlApp
from firecrawl.v2.types import JsonFormat
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Extract data into a Cloud Database(Neon Postgresql)

In [3]:
# Connect to Cloud-DB (Postgresql)

# Importing Secret keys
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
DATABASE_URL = user_secrets.get_secret("DATABASE_URL")
GEMINI_API = user_secrets.get_secret("GEMINI_API_KEY")
GEMINI_MODEL = user_secrets.get_secret("GEMINI_FLASH_MODEL")
FIRECRAWL_API = user_secrets.get_secret("FIRECRAWL_API")

In [4]:
# ── Kill Switch ───────────────────────────────────────────────────────────────
# Set to True ONLY when you need to re-scrape and reload the database.
# False = skip ETL entirely (DB already has the data)
ETL_PIPELINE = False

# ── Uncomment to upload to DB ─────────────────────────────────────────────────
# ETL pipeline to feed upload to the Database

app = FirecrawlApp(api_key=FIRECRAWL_API)

# ── Shared Schema & Prompt ────────────────────────────────────────────────────
# JSON Schema that defines the expected structure of extracted car listing data.
# Firecrawl uses this schema to validate and shape the scraped output.
UNIFIED_SCHEMA = {
    "type": "object",
    "properties": {
        "listings": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "make":             {"type": "string"},   # Car brand (e.g. TOYOTA)
                    "model":            {"type": "string"},   # Car model (e.g. LAND CRUISER)
                    "year":             {"type": "integer"},  # Manufacturing year
                    "mileage_km":       {"type": "string"},   # Odometer reading in km
                    "engine_cc":        {"type": "string"},   # Engine displacement in cc
                    "fuel_type":        {"type": "string"},   # e.g. Petrol, Diesel, Hybrid
                    "transmission":     {"type": "string"},   # e.g. Automatic, Manual
                    "body_type":        {"type": "string"},   # e.g. SUV, Sedan, Hatchback
                    "drive_type":       {"type": "string"},   # e.g. 4WD, 2WD, AWD
                    "stock_id":         {"type": "string"},   # Dealer's unique stock identifier
                    "price_usd":        {"type": "string"},   # Base price in USD (digits only)
                    "total_price_usd":  {"type": "string"},   # Total price including fees
                    "url":              {"type": "string"},   # Full listing URL
                },
                # These three fields are mandatory in every listing object
                "required": ["make", "model", "year"]
            }
        }
    },
    # The top-level "listings" array must always be present
    "required": ["listings"]
}

# Natural language prompt sent to the AI model alongside the schema.
# Instructs the model on how to extract and normalize each field.
UNIFIED_PROMPT = """Extract EVERY car listing on this page into the "listings" array.
Rules:
- ONLY include vehicles with year >= 2018. Skip older ones entirely.
- make: UPPERCASE (TOYOTA, HONDA, etc.)
- model: as shown (e.g. LAND CRUISER, SWIFT)
- year: integer
- mileage_km: digits + commas only (convert miles to km if needed)
- engine_cc: numeric only (e.g. 1500)
- fuel_type, transmission, body_type: clean and consistent
- price_usd / total_price_usd: digits only
- url: full absolute URL
Be accurate and complete."""

# ── Site Configurations ───────────────────────────────────────────────────────
# Dictionary mapping each source site name to its scraping config.
# "url_fn" is a lambda that generates the paginated URL for a given page number.
SITE_CONFIGS = {
    "SBT Japan": {
        "source": "SBT Japan",
        "url_fn": lambda p: f"https://www.sbtjapan.com/used-cars/search?year__gte=2018&page={p}",
    },
    "Car From Japan": {
        "source": "Car From Japan",
        "url_fn": lambda p: f"https://carfromjapan.com/cheap-used-cars-for-sale?maxYear=2026&minYear=2018&page={p}",
    },
    "AAA Japan": {
        "source": "AAA Japan",
        "url_fn": lambda p: f"https://www.aaajapan.com/stock-list/?min_year=2018&pg={p}",
    },
    "JCT": {
        "source": "JCT",
        "url_fn": lambda p: f"https://www.japanesecartrade.com/cars/search?year_min=2018&p={p}",
    },
    "BE FORWARD": {
        "source": "BE FORWARD",
        "url_fn": lambda p: f"https://www.beforward.jp/stocklist/mfg_year_from=2018/page={p}/",
    },
}

# ── Scrape One Page ───────────────────────────────────────────────────────────
def scrape_page(url: str, source: str, timeout_ms: int = 90000) -> list[dict]:
    """
    Scrapes a single paginated URL using Firecrawl's structured JSON extraction.

    Args:
        url:        The full URL of the page to scrape.
        source:     Human-readable name of the source site (e.g. "SBT Japan").
        timeout_ms: Max wait time in milliseconds before the scrape request times out.

    Returns:
        A list of car listing dicts, each tagged with 'source_platform'.
        Returns an empty list on failure.
    """
    print(f" GET → {url}")
    try:
        # Request Firecrawl to scrape the URL and extract JSON matching UNIFIED_SCHEMA
        doc = app.scrape(
            url,
            formats=[JsonFormat(type="json", schema=UNIFIED_SCHEMA, prompt=UNIFIED_PROMPT)],
            timeout=timeout_ms,
        )

        # Safely extract the "listings" array from the returned JSON payload
        raw = getattr(doc, "json", None) or {}
        listings = (
            raw.get("listings", []) if isinstance(raw, dict)
            else (raw if isinstance(raw, list) else [])
        )

        # Tag every listing with its originating source platform
        for rec in listings:
            rec["source_platform"] = source

        print(f"   ✓ {len(listings)} listings extracted")
        return listings

    except Exception as e:
        # Log the error and return an empty list so the pipeline continues
        print(f"   ✗ ERROR: {e}")
        return []


# ── Clean DataFrame ───────────────────────────────────────────────────────────
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans and normalises a raw listings DataFrame.

    Steps:
      1. Filters out vehicles older than 2018.
      2. Strips non-numeric characters from price and engine_cc columns.
      3. Strips non-numeric/comma characters from mileage_km.
      4. Reorders columns to match the database schema.

    Args:
        df: Raw DataFrame built from scraped listing dicts.

    Returns:
        A cleaned DataFrame with only the desired columns (in order).
    """
    if df.empty:
        return df

    # ── Step 1: Year filter ───────────────────────────────────────────────────
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")  # Coerce bad values to NaN
        df = df[df["year"] >= 2018].copy()                        # Drop pre-2018 rows

    # ── Step 2: Numeric-only columns ──────────────────────────────────────────
    # Remove everything except digits from price and engine displacement fields
    for col in ["price_usd", "total_price_usd", "engine_cc"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(r"[^\d]", "", regex=True)  # Keep digits only
                .replace({"": None, "nan": None})        # Treat empty strings as NULL
            )

    # ── Step 3: Mileage field ─────────────────────────────────────────────────
    # Keep digits and commas (e.g. "45,000") but strip units like "km" or "miles"
    if "mileage_km" in df.columns:
        df["mileage_km"] = (
            df["mileage_km"]
            .astype(str)
            .str.replace(r"[^\d,]", "", regex=True)
            .replace({"": None, "nan": None})
        )

    # ── Step 4: Column selection & ordering ───────────────────────────────────
    # Only keep columns that exist in the DataFrame (avoids KeyErrors)
    desired = [
        "make", "model", "year", "mileage_km", "engine_cc", "fuel_type",
        "transmission", "body_type", "drive_type", "stock_id",
        "price_usd", "total_price_usd", "url", "source_platform"
    ]
    return df[[c for c in desired if c in df.columns]]


# ── Database Helpers ──────────────────────────────────────────────────────────

def get_engine():
    """
    Creates and returns a SQLAlchemy engine connected to the target database.
    NullPool is used to avoid connection pooling issues in short-lived scripts.
    """
    return create_engine(DATABASE_URL, poolclass=NullPool)


def ensure_table(engine):
    """
    Creates the 'car_listings' table if it does not already exist.
    The UNIQUE constraint on (stock_id, source_platform) prevents duplicate rows
    from the same dealer listing being inserted more than once.
    """
    ddl = """CREATE TABLE IF NOT EXISTS car_listings (
        id               SERIAL PRIMARY KEY,
        make             TEXT,
        model            TEXT,
        year             INTEGER,
        mileage_km       TEXT,
        engine_cc        TEXT,
        fuel_type        TEXT,
        transmission     TEXT,
        body_type        TEXT,
        drive_type       TEXT,
        stock_id         TEXT,
        price_usd        BIGINT,
        total_price_usd  BIGINT,
        url              TEXT,
        source_platform  TEXT,
        scraped_at       TIMESTAMPTZ DEFAULT NOW(),
        UNIQUE (stock_id, source_platform)
    );"""
    with engine.begin() as conn:
        conn.execute(text(ddl))
    print("✓ Table ready\n")


def upsert_df(df: pd.DataFrame, engine) -> int:
    """
    Inserts rows from the DataFrame into the database.
    Uses ON CONFLICT DO NOTHING to silently skip already-existing listings
    (matched by the unique key: stock_id + source_platform).

    Args:
        df:     Cleaned DataFrame to insert.
        engine: Active SQLAlchemy engine.

    Returns:
        Total number of newly inserted rows.
    """
    if df.empty:
        return 0

    # Convert price columns to nullable Int64 so NaN maps to SQL NULL (not 0)
    for col in ["price_usd", "total_price_usd"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    cols = list(df.columns)

    # Build a parameterised INSERT statement using SQLAlchemy named bind params
    insert_sql = text(f"""
        INSERT INTO car_listings ({", ".join(cols)})
        VALUES ({", ".join([f":{c}" for c in cols])})
        ON CONFLICT (stock_id, source_platform) DO NOTHING;
    """)

    # Convert DataFrame rows to dicts; replace pandas NA/NaN with Python None
    rows = [
        {k: (None if pd.isna(v) else v) for k, v in row.items()}
        for row in df.to_dict("records")
    ]

    inserted = 0
    with engine.begin() as conn:
        for row in rows:
            # Fallback stock_id for listings that lack one (prevents NULL unique-key conflicts)
            if not row.get("stock_id"):
                row["stock_id"] = f"NO_ID_{row.get('source_platform', 'X')}_{inserted}"
            result = conn.execute(insert_sql, row)
            inserted += result.rowcount

    return inserted


# ── MAIN ETL FUNCTION (Auto Pagination) ───────────────────────────────────────
def etl_pipeline(
    sites: list[str] | None = None,
    max_pages_per_site: int = 50,   # Hard upper limit to prevent runaway scraping
    sleep_secs: int = 3,            # Polite delay between page requests
    save_csv: bool = True,          # Whether to export a combined CSV at the end
) -> pd.DataFrame:
    """
    Full Extract → Transform → Load pipeline.

    For each configured site:
      1. Paginates through listings pages until no results are returned or the
         page cap is hit.
      2. Cleans and normalises the raw data.
      3. Upserts clean rows into the PostgreSQL database.

    Finally, concatenates all site data into one combined DataFrame,
    optionally saves it as a CSV, and prints a summary report.

    Args:
        sites:              List of site keys to scrape (defaults to all in SITE_CONFIGS).
        max_pages_per_site: Maximum pages to scrape per site before stopping.
        sleep_secs:         Seconds to wait between consecutive page requests.
        save_csv:           If True, saves the combined results to 'japan_cars_combined.csv'.

    Returns:
        Combined DataFrame of all scraped and cleaned listings.
    """
    # Default to scraping all configured sites if no specific list is provided
    sites = sites or list(SITE_CONFIGS.keys())

    engine = get_engine()
    ensure_table(engine)   # Create DB table if it doesn't exist yet
    all_frames = []        # Accumulates per-site DataFrames for final concatenation

    for site_key in sites:
        cfg = SITE_CONFIGS[site_key]
        print(f"\n{'='*60}")
        print(f"START SCRAPING: {site_key}  (until no more results)")
        print(f"{'='*60}")

        raw_listings = []
        page = 1

        # ── Pagination loop ───────────────────────────────────────────────────
        # Keeps fetching the next page until either:
        #   a) The page returns no listings (end of results), or
        #   b) The max_pages_per_site safety cap is reached
        while page <= max_pages_per_site:
            url = cfg["url_fn"](page)
            listings = scrape_page(url, cfg["source"])

            if not listings:
                # Empty page signals we've exhausted this site's results
                print(f"   → No more listings on page {page}. Stopping {site_key}.\n")
                break

            raw_listings.extend(listings)
            page += 1
            time.sleep(sleep_secs)   # Throttle requests to be respectful to the server

        print(f" Total raw listings from {site_key}: {len(raw_listings)}")

        if not raw_listings:
            continue   # Skip to next site if nothing was collected

        # ── Transform ─────────────────────────────────────────────────────────
        df = clean_df(pd.DataFrame(raw_listings))
        print(f" Clean listings (>=2018): {len(df)}")

        # ── Load ──────────────────────────────────────────────────────────────
        try:
            n = upsert_df(df.copy(), engine)
            print(f" ✓ Inserted {n} new records into database")
        except Exception as e:
            print(f" ✗ DB Error: {e}")

        all_frames.append(df)

    # ── Final Output ──────────────────────────────────────────────────────────
    if not all_frames:
        print("\n⚠ No data collected.")
        return pd.DataFrame()

    # Merge all per-site DataFrames into one
    combined = pd.concat(all_frames, ignore_index=True)

    # Optionally persist the combined result as a local CSV file
    if save_csv:
        combined.to_csv("japan_cars_combined.csv", index=False)
        print(f"\n✅ Combined CSV saved → japan_cars_combined.csv")

    # ── Summary Report ────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print(f"Total listings : {len(combined)}")
    print(f"By source      : {combined['source_platform'].value_counts().to_dict()}")

    if "year" in combined.columns:
        print(f"Years          : {int(combined['year'].min())} – {int(combined['year'].max())}")

    if "price_usd" in combined.columns:
        filled = combined["price_usd"].notna().sum()
        print(f"Price filled   : {filled}/{len(combined)} ({filled/len(combined)*100:.1f}%)")

    print(f"{'='*60}\n")

    return combined


# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    if ETL_PIPELINE:
        # Kill switch is ON — run the full scrape and DB load
        df = etl_pipeline(max_pages_per_site=50)
        # df = etl_pipeline(sites=["SBT Japan"], max_pages_per_site=30)  # Single-site debug run
    else:
        # Kill switch is OFF — DB already populated, skip scraping entirely
        print("⏭ ETL skipped — DB already populated. Set ETL_PIPELINE = True to re-run.")

⏭ ETL skipped — DB already populated. Set ETL_PIPELINE = True to re-run.


In [5]:
# Confirm Database Connection and the rows 
from sqlalchemy import create_engine, text

# Create DB connection
engine = create_engine(DATABASE_URL)

# Check connection + total records
with engine.connect() as conn:

    print("✅ Database Connected Successfully\n")

    # Get all tables
    tables = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema='public';
    """)).fetchall()

    for table in tables:

        table_name = table[0]

        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table_name};")
        ).scalar()

        print(f"📦 {table_name}: {count} records")

✅ Database Connected Successfully

📦 car_listings: 1085 records


# Clean the data, and prepare it for analysis.

In [6]:
# ============================================================
# LOAD DATA FROM POSTGRESQL INTO DATAFRAME
# ============================================================


# Create engine
engine = create_engine(DATABASE_URL)

# Read full table into dataframe
df = pd.read_sql("SELECT * FROM car_listings;", engine)

# Basic inspection
print("Shape:", df.shape)

display(df.head())

print("\nColumns:\n")
print(df.columns.tolist())

print("\nData Types:\n")
print(df.dtypes)

print("\nMissing Values:\n")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (1085, 16)


,id,make,model,year,mileage_km,engine_cc,fuel_type,transmission,body_type,drive_type,stock_id,price_usd,total_price_usd,url,source_platform,scraped_at
0,1,SUZUKI,SWIFT,2022,"83,000",1242,PETROL,AT,Hatchback,2WD,AH6386,6000.0,7462.0,https://www.sbtjapan.com/used-cars/AH6386,SBT Japan,2026-05-14 17:20:11.692332+00:00
1,2,HONDA,VEZEL,2021,"64,000",1500,HYBRID,AT,SUV,2WD,AI6460,NaN,NaN,https://www.sbtjapan.com/used-cars/AI6460,SBT Japan,2026-05-14 17:20:11.692332+00:00
2,3,HONDA,VEZEL,2021,"40,000",1500,HYBRID,AT,SUV,2WD,AI5717,13900.0,15930.0,https://www.sbtjapan.com/used-cars/AI5717,SBT Japan,2026-05-14 17:20:11.692332+00:00
3,4,HONDA,SHUTTLE,2020,"18,000",1500,HYBRID,AT,Hatchback,2WD,AI9330,5280.0,6699.0,https://www.sbtjapan.com/used-cars/AI9330,SBT Japan,2026-05-14 17:20:11.692332+00:00
4,5,TOYOTA,COROLLA AXIO,2019,"63,000",1300,PETROL,AT,Sedan,2WD,AI9250,6880.0,8297.0,https://www.sbtjapan.com/used-cars/AI9250,SBT Japan,2026-05-14 17:20:11.692332+00:00



Columns:

['id', 'make', 'model', 'year', 'mileage_km', 'engine_cc', 'fuel_type', 'transmission', 'body_type', 'drive_type', 'stock_id', 'price_usd', 'total_price_usd', 'url', 'source_platform', 'scraped_at']

Data Types:

id                               int64
make                            object
model                           object
year                             int64
mileage_km                      object
engine_cc                       object
fuel_type                       object
transmission                    object
body_type                       object
drive_type                      object
stock_id                        object
price_usd                      float64
total_price_usd                float64
url                             object
source_platform                 object
scraped_at         datetime64[ns, UTC]
dtype: object

Missing Values:

id                   0
make                 0
model                0
year                 0
mileage_km           3
engin

## Handling Missing Values

In [7]:
# ============================================================
# PRICE COMPLETENESS ANALYSIS
# ============================================================

# both prices exist
both_prices = df[
    df['price_usd'].notnull() &
    df['total_price_usd'].notnull()
]

# only price_usd exists
only_price = df[
    df['price_usd'].notnull() &
    df['total_price_usd'].isnull()
]

# only total_price_usd exists
only_total = df[
    df['price_usd'].isnull() &
    df['total_price_usd'].notnull()
]

# neither exists
no_prices = df[
    df['price_usd'].isnull() &
    df['total_price_usd'].isnull()
]

print("=== SUMMARY ===")
print("Both prices present:", len(both_prices))
print("Only price_usd:", len(only_price))
print("Only total_price_usd:", len(only_total))
print("No prices at all:", len(no_prices))

=== SUMMARY ===
Both prices present: 886
Only price_usd: 179
Only total_price_usd: 0
No prices at all: 20


In [8]:
# ============================================================
# INSPECT SAMPLES
# ============================================================

display(both_prices[['make','model','year','price_usd','total_price_usd','url']].head(10))
display(only_price[['make','model','year','price_usd','url']].head(10))
display(only_total[['make','model','year','total_price_usd','url']].head(10))
display(no_prices[['make','model','year','url']].head(10))

,make,model,year,price_usd,total_price_usd,url
0,SUZUKI,SWIFT,2022,6000.0,7462.0,https://www.sbtjapan.com/used-cars/AH6386
2,HONDA,VEZEL,2021,13900.0,15930.0,https://www.sbtjapan.com/used-cars/AI5717
3,HONDA,SHUTTLE,2020,5280.0,6699.0,https://www.sbtjapan.com/used-cars/AI9330
4,TOYOTA,COROLLA AXIO,2019,6880.0,8297.0,https://www.sbtjapan.com/used-cars/AI9250
5,TOYOTA,COROLLA AXIO,2019,6200.0,7821.0,https://www.sbtjapan.com/used-cars/AI9329
6,MAZDA,DEMIO,2019,6260.0,7879.0,https://www.sbtjapan.com/used-cars/AF4765
7,TOYOTA,VOXY,2018,10330.0,12591.0,https://www.sbtjapan.com/used-cars/AI6388
8,TOYOTA,SUCCEED VAN UL,2019,6700.0,8327.0,https://www.sbtjapan.com/used-cars/AH5354
10,TOYOTA,AQUA,2018,3950.0,5210.0,https://www.sbtjapan.com/used-cars/AI7392
11,TOYOTA,AQUA,2018,3950.0,5392.0,https://www.sbtjapan.com/used-cars/AI7393


,make,model,year,price_usd,url
58,DAIHATSU,HIJET TRUCK,2022,6831.0,https://carfromjapan.com/cheap-used-daihatsu-h...
129,TOYOTA,C-HR G T,2019,13500.0,https://www.sbtjapan.com/used-cars/AI4805
130,TOYOTA,COROLLA FIELDER HYBRID G,2019,7500.0,https://www.sbtjapan.com/used-cars/AI4871
131,TOYOTA,PROBOX VAN HYBRID GX,2023,6300.0,https://www.sbtjapan.com/used-cars/AI4630
132,MAZDA,CX5 25S PROACTIVE,2021,13000.0,https://www.sbtjapan.com/used-cars/AI4776
133,TOYOTA,HARRIER HYBRID Z,2022,15950.0,https://www.sbtjapan.com/used-cars/AI4723
134,TOYOTA,VITZ F,2018,3900.0,https://www.sbtjapan.com/used-cars/AH8678
135,TOYOTA,TANK CUSTOM G,2019,4360.0,https://www.sbtjapan.com/used-cars/AI3717
136,NISSAN,NV200VANETTE VAN DX,2019,5400.0,https://www.sbtjapan.com/used-cars/AH8893
137,NISSAN,NV200VANETTE VAN DX,2019,5600.0,https://www.sbtjapan.com/used-cars/AI2789


,make,model,year,total_price_usd,url


,make,model,year,url
1,HONDA,VEZEL,2021,https://www.sbtjapan.com/used-cars/AI6460
9,MAZDA,CX3,2019,https://www.sbtjapan.com/used-cars/AH5547
106,TOYOTA,YARIS X,2021,https://www.sbtjapan.com/used-cars/AI6309
109,HONDA,CIVIC SEDAN,2018,https://www.sbtjapan.com/used-cars/AI6532
114,TOYOTA,ALPHARD,2019,https://www.sbtjapan.com/used-cars/AI4567
115,SUZUKI,SWIFT HYBRID RS,2019,https://www.sbtjapan.com/used-cars/AI1391
118,HONDA,CRV HYBRID EX MASTERPIECE,2019,https://www.sbtjapan.com/used-cars/AI1131
159,HONDA,VEZEL G,2018,https://www.sbtjapan.com/used-cars/AI2938
188,DAIHATSU,TANTO L LTD SA3,2019,https://www.sbtjapan.com/used-cars/AI2278
194,TOYOTA,PROBOX VAN DX COMFORT,2021,https://www.sbtjapan.com/used-cars/AH9282


In [9]:
## Data Integrity Triage
## Rows where price_usd AND total_price_usd are missing
## → then see if those rows also have missing mileage_km or engine_cc

# ============================================================
# NO PRICE ROWS
# ============================================================

no_prices = df[
    df['price_usd'].isnull() &
    df['total_price_usd'].isnull()
]

print("No price rows:", len(no_prices))
display(no_prices[['make','model','year','mileage_km','engine_cc','url']])

No price rows: 20


,make,model,year,mileage_km,engine_cc,url
1,HONDA,VEZEL,2021,"64,000",1500,https://www.sbtjapan.com/used-cars/AI6460
9,MAZDA,CX3,2019,"62,000",1997,https://www.sbtjapan.com/used-cars/AH5547
106,TOYOTA,YARIS X,2021,"2,000",1000,https://www.sbtjapan.com/used-cars/AI6309
109,HONDA,CIVIC SEDAN,2018,"77,000",1500,https://www.sbtjapan.com/used-cars/AI6532
114,TOYOTA,ALPHARD,2019,"232,000",2400,https://www.sbtjapan.com/used-cars/AI4567
115,SUZUKI,SWIFT HYBRID RS,2019,"59,000",1200,https://www.sbtjapan.com/used-cars/AI1391
118,HONDA,CRV HYBRID EX MASTERPIECE,2019,"71,000",2000,https://www.sbtjapan.com/used-cars/AI1131
159,HONDA,VEZEL G,2018,"8,000",1500,https://www.sbtjapan.com/used-cars/AI2938
188,DAIHATSU,TANTO L LTD SA3,2019,"159,000",660,https://www.sbtjapan.com/used-cars/AI2278
194,TOYOTA,PROBOX VAN DX COMFORT,2021,"62,000",1300,https://www.sbtjapan.com/used-cars/AH9282


## Strict dataset integrity control


* remove 20 rows with no price
* remove 3 rows missing mileage
* remove 1 row missing engine_cc
* keep a single unified price column


In [10]:
# Have one clean price column
df['final_price_usd'] = df['price_usd'].fillna(df['total_price_usd'])

In [11]:
# Remove all invalid rows
# A. Remove rows with no price at all (20 rows)
# B. Remove missing mileage (3 rows)
# C. Remove missing engine_cc (1 row)

complete_df = df[
    df['final_price_usd'].notnull() &
    df['mileage_km'].notnull() &
    df['engine_cc'].notnull()
].copy()

In [12]:
# Verify its removed
print("Original shape:", df.shape)
print("Complete dataset shape:", complete_df.shape)

print("\nRemoved rows:", df.shape[0] - complete_df.shape[0])

Original shape: (1085, 17)
Complete dataset shape: (1061, 17)

Removed rows: 24


In [13]:
# Ensure only one price column exists
complete_df = complete_df.drop(
    columns=['price_usd', 'total_price_usd'],
    errors='ignore'
)

# rename unified target
complete_df = complete_df.rename(
    columns={'final_price_usd': 'price_usd'}
)

In [14]:
complete_df.head()

,id,make,model,year,mileage_km,engine_cc,fuel_type,transmission,body_type,drive_type,stock_id,url,source_platform,scraped_at,price_usd
0,1,SUZUKI,SWIFT,2022,"83,000",1242,PETROL,AT,Hatchback,2WD,AH6386,https://www.sbtjapan.com/used-cars/AH6386,SBT Japan,2026-05-14 17:20:11.692332+00:00,6000.0
2,3,HONDA,VEZEL,2021,"40,000",1500,HYBRID,AT,SUV,2WD,AI5717,https://www.sbtjapan.com/used-cars/AI5717,SBT Japan,2026-05-14 17:20:11.692332+00:00,13900.0
3,4,HONDA,SHUTTLE,2020,"18,000",1500,HYBRID,AT,Hatchback,2WD,AI9330,https://www.sbtjapan.com/used-cars/AI9330,SBT Japan,2026-05-14 17:20:11.692332+00:00,5280.0
4,5,TOYOTA,COROLLA AXIO,2019,"63,000",1300,PETROL,AT,Sedan,2WD,AI9250,https://www.sbtjapan.com/used-cars/AI9250,SBT Japan,2026-05-14 17:20:11.692332+00:00,6880.0
5,6,TOYOTA,COROLLA AXIO,2019,"108,000",1500,HYBRID,AT,Sedan,2WD,AI9329,https://www.sbtjapan.com/used-cars/AI9329,SBT Japan,2026-05-14 17:20:11.692332+00:00,6200.0


# Feature Engineering

In [15]:
df = complete_df.copy()

## 1. CAR AGE FEATURE

In [16]:
current_year = 2026

df['car_age'] = current_year - df['year']

# safety check (no negative ages)
df['car_age'] = df['car_age'].clip(lower=0)

df[['year', 'car_age']].head()

,year,car_age
0,2022,4
2,2021,5
3,2020,6
4,2019,7
5,2019,7


## 2. MILEAGE PER YEAR

In [17]:
df['mileage_km'] = (
    df['mileage_km']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('None', '', regex=False)
    .str.replace('nan', '', regex=False)
    .str.strip()
)

df['mileage_km'] = pd.to_numeric(df['mileage_km'], errors='coerce')

df['car_age'] = pd.to_numeric(df['car_age'], errors='coerce')

df['mileage_per_year'] = df['mileage_km'] / (df['car_age'] + 1)

In [18]:
print(df[['mileage_km', 'car_age']].dtypes)
print(df[['mileage_km', 'car_age']].isnull().sum())

mileage_km    int64
car_age       int64
dtype: object
mileage_km    0
car_age       0
dtype: int64


In [19]:
df = df.copy()

df['mileage_km'] = pd.to_numeric(df['mileage_km'], errors='coerce')
df['car_age'] = pd.to_numeric(df['car_age'], errors='coerce')

df['mileage_per_year'] = df['mileage_km'] / (df['car_age'] + 1)

## 3. ENGINE CLASSIFICATION

In [20]:
df['engine_cc'].apply(type).value_counts()

engine_cc
<class 'str'>    1061
Name: count, dtype: int64

In [21]:
df['engine_cc'] = (
    df['engine_cc']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('cc', '', regex=False)
    .str.strip()
)

df['engine_cc'] = pd.to_numeric(df['engine_cc'], errors='coerce')

In [22]:
df['engine_cc'] = df['engine_cc'].astype('float').astype('Int64')

In [23]:
def engine_class(cc):
    if pd.isna(cc):
        return "unknown"
    if cc <= 660:
        return "kei"
    elif cc <= 1500:
        return "small"
    elif cc <= 2500:
        return "medium"
    else:
        return "large"

In [24]:
df['engine_class'] = df['engine_cc'].apply(engine_class)

In [25]:
df[['engine_cc', 'engine_class']].head(20)

,engine_cc,engine_class
0,1242,small
2,1500,small
3,1500,small
4,1300,small
5,1500,small
6,1490,small
7,2000,medium
8,1500,small
10,1500,small
11,1500,small


## 4. VEHICLE SEGMENT FEATURES (MARKET VALUE CORE)

In [26]:
# normalize body type text
df['segment'] = (
    df['body_type']
    .astype(str)
    .str.lower()
    .str.strip()
)

# segment flags
df['is_suv'] = df['segment'].str.contains('suv', na=False)
df['is_van'] = df['segment'].str.contains('van', na=False)
df['is_hatchback'] = df['segment'].str.contains('hatch', na=False)

# inspect output
df[['body_type', 'segment', 'is_suv', 'is_van', 'is_hatchback']].head(20)

,body_type,segment,is_suv,is_van,is_hatchback
0,Hatchback,hatchback,False,False,True
2,SUV,suv,True,False,False
3,Hatchback,hatchback,False,False,True
4,Sedan,sedan,False,False,False
5,Sedan,sedan,False,False,False
6,Hatchback,hatchback,False,False,True
7,Minivan,minivan,False,True,False
8,Van,van,False,True,False
10,Hatchback,hatchback,False,False,True
11,Hatchback,hatchback,False,False,True


In [27]:
# validate distributions
print("SUVs:")
print(df['is_suv'].value_counts())

print("\nVans:")
print(df['is_van'].value_counts())

print("\nHatchbacks:")
print(df['is_hatchback'].value_counts())

SUVs:
is_suv
False    737
True     324
Name: count, dtype: int64

Vans:
is_van
False    854
True     207
Name: count, dtype: int64

Hatchbacks:
is_hatchback
False    991
True      70
Name: count, dtype: int64


## 5 PRICE NORMALIZATION (ML STABILITY LAYER)

In [28]:
df['log_price'] = np.log1p(df['price_usd'])

df[['price_usd', 'log_price']].head()

,price_usd,log_price
0,6000.0,8.699681
2,13900.0,9.539716
3,5280.0,8.571871
4,6880.0,8.836519
5,6200.0,8.732466


# ML Modeling Stage

In [29]:
from sklearn.model_selection import train_test_split

In [30]:
df.columns

Index(['id', 'make', 'model', 'year', 'mileage_km', 'engine_cc', 'fuel_type',
       'transmission', 'body_type', 'drive_type', 'stock_id', 'url',
       'source_platform', 'scraped_at', 'price_usd', 'car_age',
       'mileage_per_year', 'engine_class', 'segment', 'is_suv', 'is_van',
       'is_hatchback', 'log_price'],
      dtype='object')

## 1. Selecting Features

In [31]:
features = [
    'make',
    'model',
    'year',
    'car_age',
    'mileage_km',
    'mileage_per_year',
    'engine_cc',
    'engine_class',
    'fuel_type',
    'transmission',
    'body_type',
    'drive_type',
    'source_platform',
    'is_suv',
    'is_van',
    'is_hatchback'
]

target = 'log_price'

In [32]:
# High impact improvement
df['make_model'] = df['make'] + "_" + df['model']
features.append('make_model')

## 2. Building X and y

In [33]:
X = df[features]
y = df[target]

## 3. Train/Test Split

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 4. Define Categorical Features

In [35]:
categorical_features = [
    'make',
    'model',
    'engine_class',
    'fuel_type',
    'transmission',
    'body_type',
    'drive_type',
    'source_platform',
    'make_model'
]

## 5. Install CATBOOST

In [36]:
!pip install catboost -q

## 6. Train Model

In [37]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose=100,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	learn: 0.9051997	test: 0.7489889	best: 0.7489889 (0)	total: 68.6ms	remaining: 1m 8s
100:	learn: 0.4021439	test: 0.2927300	best: 0.2927300 (100)	total: 1.06s	remaining: 9.45s
200:	learn: 0.3155131	test: 0.2818935	best: 0.2818363 (199)	total: 1.99s	remaining: 7.89s
300:	learn: 0.2572933	test: 0.2760944	best: 0.2759446 (299)	total: 3.05s	remaining: 7.09s
400:	learn: 0.2126241	test: 0.2723703	best: 0.2723494 (399)	total: 4.18s	remaining: 6.25s
500:	learn: 0.1814191	test: 0.2713305	best: 0.2703747 (451)	total: 5.28s	remaining: 5.26s
600:	learn: 0.1575842	test: 0.2695254	best: 0.2695254 (600)	total: 6.37s	remaining: 4.23s
700:	learn: 0.1393407	test: 0.2683940	best: 0.2681222 (683)	total: 7.47s	remaining: 3.19s
800:	learn: 0.1232062	test: 0.2673406	best: 0.2671543 (762)	total: 8.57s	remaining: 2.13s
900:	learn: 0.1101780	test: 0.2676182	best: 0.2670140 (818)	total: 9.66s	remaining: 1.06s
999:	learn: 0.0996500	test: 0.2676662	best: 0.2670140 (818)	total: 10.8s	remaining: 0us

bestTest = 0.2

CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=1000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

## 7. Predict

In [38]:
pred_log = model.predict(X_test)

## 8. Convert Back to USD

In [39]:
pred_price = np.expm1(pred_log)
actual_price = np.expm1(y_test)

## 9. Evaluate Model

In [40]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(actual_price, pred_price)
r2 = r2_score(actual_price, pred_price)

print("MAE:", round(mae, 2))
print("R²:", round(r2, 4))

MAE: 3564.12
R²: 0.7759


### Model Perfomance

#### R² = 0.7541

* 75% of price variation is captured by the features. ***(Scraped automative data)***
* In real market data, **0.70–0.80** is solid production baseline.

#### MAE = 3564 USD
* Average absolute error ≈ $3.5k per vehicle
* In the dataset cars range from ~$3k → $30k+
* so error is roughly 10–20% band for many vehicles

**Acceptable for market estimation, not yet precise enough for pricing arbitration**

## 10. Inspect Predictions

In [41]:
results = X_test.copy()

results['actual_price'] = actual_price
results['predicted_price'] = pred_price

results[['make', 'model', 'actual_price', 'predicted_price']].head(20)

,make,model,actual_price,predicted_price
33,TOYOTA,COROLLA CROSS,15500.0,18261.024251
434,TOYOTA,HILUX,26299.0,25918.412248
556,SUZUKI,JIMNY,12961.0,13775.650262
764,SUZUKI,JIMNY,15737.0,15522.676771
813,NISSAN,FAIRLADY Z,32238.0,39453.441070
589,DAIHATSU,HIJET TRUCK,11172.0,9371.950269
741,DAIHATSU,HIJET CARGO,6253.0,8668.459650
599,TOYOTA,LAND CRUISER PRADO,26741.0,32580.041803
89,TOYOTA,LAND CRUISER,58249.0,51262.411725
665,SUBARU,FORESTER,40148.0,32673.377346


# Import Cost Engine

### 1. Base Input

In [42]:
df_results = results.copy()  # X_test + actual + predicted
df_results['predicted_price_usd'] = df_results['predicted_price']

### 2. CIF Estimation

In [43]:
SHIPPING_USD = 1200  # rough Japan → Mombasa container share
INSURANCE_RATE = 0.002  # 0.2%

def compute_cif(fob):
    # CIF = FOB + shipping + insurance approximation
    return fob + SHIPPING_USD + (fob * INSURANCE_RATE)

df_results['CIF'] = df_results['predicted_price_usd'].apply(compute_cif)

### 3. KRA TAX ENGINE

In [44]:
def kra_taxes(row):
    cif = row['CIF']
    engine_cc = row['engine_cc'] if 'engine_cc' in row else 1500

    # Import Duty
    import_duty = 0.25 * cif

    # Excise Duty
    if engine_cc <= 1500:
        excise = 0.20 * (cif + import_duty)
    else:
        excise = 0.25 * (cif + import_duty)

    # VAT
    vat = 0.16 * (cif + import_duty + excise)

    # RDL
    rdl = 0.02 * cif

    # IDF
    idf = 0.035 * cif

    return import_duty + excise + vat + rdl + idf

df_results['KRA_taxes'] = df_results.apply(kra_taxes, axis=1)

### 4. PORT + CLEARING FEES

In [45]:
def clearing_cost(row):
    base = 200000  # midpoint estimate (KES)
    return base

df_results['clearing_cost_kes'] = df_results.apply(clearing_cost, axis=1)

### 5. Landing Cost (Final Import Cost)

In [46]:
!pip install forex-python

In [47]:
import requests

def get_rate():
    url = "https://open.er-api.com/v6/latest/USD"
    data = requests.get(url, timeout=10).json()
    return data["rates"]["KES"]

In [48]:
rate = get_rate()

df_results['landed_cost_kes'] = (
    df_results['CIF'] * rate +
    df_results['KRA_taxes'] +
    df_results['clearing_cost_kes']
)

### 6. Kenyan Market Comparison

In [49]:
rate = get_rate()

df_results['kenya_market_price_kes'] = (
    df_results['actual_price'] * rate
)

### 7. Savings/ROI Engine

In [50]:
df_results['savings_kes'] = (
    df_results['kenya_market_price_kes'] - df_results['landed_cost_kes']
)

df_results['roi_percent'] = (
    df_results['savings_kes'] / df_results['landed_cost_kes']
) * 100

### 8. Decision Logic

In [51]:
def decision(row):
    if row['savings_kes'] > 0:
        return "IMPORT (PROFITABLE)"
    else:
        return "DO NOT IMPORT"

df_results['decision'] = df_results.apply(decision, axis=1)

### 9. Output View

In [52]:
df_results[[
    'make',
    'model',
    'actual_price',
    'predicted_price_usd',
    'landed_cost_kes',
    'kenya_market_price_kes',
    'savings_kes',
    'roi_percent',
    'decision'
]].head(20)

,make,model,actual_price,predicted_price_usd,landed_cost_kes,kenya_market_price_kes,savings_kes,roi_percent,decision
33,TOYOTA,COROLLA CROSS,15500.0,18261.024251,2.735696e+06,2.002361e+06,-7.333354e+05,-26.806171,DO NOT IMPORT
434,TOYOTA,HILUX,26299.0,25918.412248,3.733547e+06,3.397425e+06,-3.361221e+05,-9.002756,DO NOT IMPORT
556,SUZUKI,JIMNY,12961.0,13775.650262,2.150110e+06,1.674361e+06,-4.757485e+05,-22.126707,DO NOT IMPORT
764,SUZUKI,JIMNY,15737.0,15522.676771,2.377642e+06,2.032978e+06,-3.446640e+05,-14.496046,DO NOT IMPORT
813,NISSAN,FAIRLADY Z,32238.0,39453.441070,5.497326e+06,4.164652e+06,-1.332674e+06,-24.242222,DO NOT IMPORT
589,DAIHATSU,HIJET TRUCK,11172.0,9371.950269,1.576574e+06,1.443250e+06,-1.333239e+05,-8.456556,DO NOT IMPORT
741,DAIHATSU,HIJET CARGO,6253.0,8668.459650,1.484952e+06,8.077911e+05,-6.771605e+05,-45.601518,DO NOT IMPORT
599,TOYOTA,LAND CRUISER PRADO,26741.0,32580.041803,4.601639e+06,3.454525e+06,-1.147114e+06,-24.928381,DO NOT IMPORT
89,TOYOTA,LAND CRUISER,58249.0,51262.411725,7.036179e+06,7.524872e+06,4.886932e+05,6.945435,IMPORT (PROFITABLE)
665,SUBARU,FORESTER,40148.0,32673.377346,4.613801e+06,5.186502e+06,5.727008e+05,12.412775,IMPORT (PROFITABLE)


## MODEL EXPORT

### 1. Folder Structure

In [53]:
import os

os.makedirs("/kaggle/working/app/models", exist_ok=True)

### 2. Save Model

In [54]:
# We using CatBoost
model.save_model("/kaggle/working/app/models/model.cbm")

### 3. Save Feature Order

In [55]:
import json

feature_columns = X.columns.tolist()

with open("/kaggle/working/app/models/features.json", "w") as f:
    json.dump(feature_columns, f)